In [1]:
from scipy.stats import ttest_rel, wilcoxon
import numpy as np

from data_utils import load_pickle
from utils import *

/home/sairamv/anaconda3/lib/python3.8/site-packages/setuptools/distutils_patch.py:25: UserWarning: Distutils was imported before Setuptools. This usage is discouraged and may exhibit undesirable behaviors or errors. Please use Setuptools' objects directly or at least import Setuptools first.
  warnings.warn(


In [12]:
def get_per_user_metrics(user_item_scores,groudtruth,topk=5):
    per_user_scores = {}
    for uid in user_item_scores:
        # [Important] Use shuffle to break ties!!!
        ui_scores = list(user_item_scores[uid].items())
        topk_preds = heapq.nlargest(topk, ui_scores, key=lambda x: x[1]) # list of k tuples
        topk_preds = [x[0] for x in topk_preds]  # list of k <item_id>
        result = evaluate_once(topk_preds, groudtruth[uid])
        
        per_user_scores[uid] = {}
        per_user_scores[uid].update(result)
        if uid == 0:
            print("per_user scores:", per_user_scores[uid])
    return per_user_scores

In [17]:
def sig_test(file_ours, file_baseline, topk=5,
             metric_names=("ndcg@k","hit@k","recall@k","precision@k")):
    """
    Loads two pickle result files (each with 'ui_scores' and 'gt'),
    computes per-user metrics via get_per_user_metrics(...),
    and runs paired significance tests (t-test + Wilcoxon) per metric.
    """
    # 1) load and compute per-user metrics
    ours = load_pickle(file_ours)
    base = load_pickle(file_baseline)

    ours_userscores = ours["ui_scores"]
    ours_gt         = ours["gt"]
    base_userscores = base["ui_scores"]
    base_gt         = base["gt"]

    per_user_ours = get_per_user_metrics(ours_userscores, ours_gt, topk=topk)
    per_user_base = get_per_user_metrics(base_userscores, base_gt, topk=topk)

    # 2) align users
    users_a = set(per_user_ours.keys())
    users_b = set(per_user_base.keys())
    if users_a != users_b:
        common = sorted(users_a & users_b)
        missing_a = sorted(users_b - users_a)
        missing_b = sorted(users_a - users_b)
        print(f"[warn] users differ; using intersection ({len(common)} users). "
              f"missing_in_ours={len(missing_a)}, missing_in_base={len(missing_b)}")
        users = common
    else:
        users = sorted(users_a)

    print(f"Users: {len(users)} | K={topk}")
    print("metric names: ",metric_names)

    # 3) run paired tests for each requested metric (if present)
    for m in metric_names:
        # skip metrics that aren't present
        if not all(m in per_user_ours[u] for u in users) or not all(m in per_user_base[u] for u in users):
            print("skipping metric ",m)
            continue

        a = np.array([per_user_ours[u][m] for u in users], dtype=float)
        b = np.array([per_user_base[u][m] for u in users], dtype=float)

        mean_a = float(np.nanmean(a))
        mean_b = float(np.nanmean(b))
        delta  = mean_a - mean_b

        # paired two-tailed t-test
        t_stat, t_p = ttest_rel(a, b, nan_policy="omit")

        # Wilcoxon (non-parametric); may fail if all diffs are zero
        try:
            w_stat, w_p = wilcoxon(a, b, zero_method="wilcox", alternative="two-sided")
        except ValueError:
            w_p = 1.0

        print(f"[{m}] A={mean_a:.6f} | B={mean_b:.6f} | Δ={delta:+.6f} | "
              f"t_p={t_p:.3g} | wilcoxon_p={w_p:.3g}")

In [22]:
print("BEAUTY")
sig_test("top-preds/beauty/stage-1-POS-only-EXPLS/DEEPFM-beauty-preds.pkl",
         '../top-preds/LLARA-Clean-beauty-preds.pkl',
         topk=3
        )

BEAUTY
per_user scores: {'precision@k': 0.0, 'recall@k': 0.0, 'ndcg@k': 0.0, 'hit@k': 0.0, 'ap': 0.0, 'rel': [0, 0, 0]}
per_user scores: {'precision@k': 0.0, 'recall@k': 0.0, 'ndcg@k': 0.0, 'hit@k': 0.0, 'ap': 0.0, 'rel': [0, 0, 0]}
Users: 22363 | K=3
metric names:  ('ndcg@k', 'hit@k', 'recall@k', 'precision@k')
[ndcg@k] A=0.202951 | B=0.218394 | Δ=-0.015443 | t_p=2.7e-07 | wilcoxon_p=8.47e-07
[hit@k] A=0.205697 | B=0.276126 | Δ=-0.070429 | t_p=9.99e-98 | wilcoxon_p=8.51e-97
[recall@k] A=0.205697 | B=0.276126 | Δ=-0.070429 | t_p=9.99e-98 | wilcoxon_p=8.51e-97
[precision@k] A=0.068566 | B=0.092042 | Δ=-0.023476 | t_p=9.99e-98 | wilcoxon_p=8.51e-97


In [25]:
print("YELP")
sig_test("top-preds/yelp/stage-1-POS-only-EXPLS/DEEPFM-yelp-preds.pkl",
             '../top-preds/other-server/CoLLM-FAIR_IPS-yelp-preds.pkl',
         topk=3
        )

YELP
per_user scores: {'precision@k': 0.3333333333333333, 'recall@k': 1.0, 'ndcg@k': 1.0, 'hit@k': 1.0, 'ap': 1.0, 'rel': [1, 0, 0]}
per_user scores: {'precision@k': 0.0, 'recall@k': 0.0, 'ndcg@k': 0.0, 'hit@k': 0.0, 'ap': 0.0, 'rel': [0, 0, 0]}
Users: 30431 | K=3
metric names:  ('ndcg@k', 'hit@k', 'recall@k', 'precision@k')
[ndcg@k] A=0.356537 | B=0.203458 | Δ=+0.153079 | t_p=0 | wilcoxon_p=0
[hit@k] A=0.358122 | B=0.250238 | Δ=+0.107883 | t_p=7.19e-268 | wilcoxon_p=1.29e-262
[recall@k] A=0.358122 | B=0.250238 | Δ=+0.107883 | t_p=7.19e-268 | wilcoxon_p=1.29e-262
[precision@k] A=0.119374 | B=0.083413 | Δ=+0.035961 | t_p=7.19e-268 | wilcoxon_p=1.29e-262


In [24]:
print("SPORTS")
sig_test("top-preds/sports/stage-1-POS-only-EXPLS/DEEPFM-sports-preds.pkl",
        "../top-preds/other-server/LLARA-Clean-sports-preds.pkl",
         topk=3
        )

SPORTS
per_user scores: {'precision@k': 0.0, 'recall@k': 0.0, 'ndcg@k': 0.0, 'hit@k': 0.0, 'ap': 0.0, 'rel': [0, 0, 0]}
per_user scores: {'precision@k': 0.0, 'recall@k': 0.0, 'ndcg@k': 0.0, 'hit@k': 0.0, 'ap': 0.0, 'rel': [0, 0, 0]}
Users: 35598 | K=3
metric names:  ('ndcg@k', 'hit@k', 'recall@k', 'precision@k')
[ndcg@k] A=0.213925 | B=0.137119 | Δ=+0.076805 | t_p=2.33e-169 | wilcoxon_p=5.03e-261
[hit@k] A=0.214169 | B=0.174869 | Δ=+0.039300 | t_p=1.43e-39 | wilcoxon_p=1.76e-39
[recall@k] A=0.214169 | B=0.174869 | Δ=+0.039300 | t_p=1.43e-39 | wilcoxon_p=1.76e-39
[precision@k] A=0.071390 | B=0.058290 | Δ=+0.013100 | t_p=1.43e-39 | wilcoxon_p=1.76e-39
